## Exercise 3-1: Building the Option Volatility Smirk from OptionMetrics

In this exercise we build a **firm-year panel of the option-implied volatility smirk** from scratch, using the WRDS querying skills from Module 3-1.

**Background.** Equity options on the same stock and with the same maturity do not trade at the same implied volatility (IV). Plotting IV against moneyness for individual US stocks produces a downward-sloping curve — a *smirk*: deep out-of-the-money (OTM) puts trade at systematically higher implied volatilities than at-the-money (ATM) calls. [Xing, Zhang and Zhao (2010, *JFQA*)](https://doi.org/10.1017/S0022109010000463) argue that the steepness of this smirk reveals what informed investors expect about the *left tail* of the return distribution: when demand for crash protection is high, OTM puts become expensive relative to ATM calls and the smirk steepens. They define

$$\text{SMIRK}_{i,t} \;=\; IV^{\text{OTM put}}_{i,t} \;-\; IV^{\text{ATM call}}_{i,t}$$

and show it predicts future stock returns and jump risk. [Kim, Li, Lu and Yu (2016, *JAE*)](https://doi.org/10.1016/j.jacceco.2015.12.003) use exactly this measure, averaged to the firm-year, as a *forward-looking, market-based* proxy for **expected crash risk**, alongside the usual accounting-based controls (size, leverage, market-to-book).

**Goal of this exercise.** Reproduce that firm-year measure. The starting point is a working SAS program, `Smirk_Daily.sas`, of the kind that circulates among AccFin researchers. Your job is to translate it into Python — which is not a line-by-line transliteration, because SAS and PostgreSQL disagree about several things that matter (see the translation table below).

The pipeline has six stages:

1. For every stock (`secid`) and every trading day, compute the **open-interest-weighted average IV** of ATM calls and of OTM puts, from OptionMetrics' daily option price file.
2. Take the difference to get the **daily smirk**.
3. Map OptionMetrics `secid` → CUSIP → CRSP `permno`.
4. Pull **Compustat** fundamentals and build the control variables.
5. Link Compustat `gvkey` → `permno` through the **CCM** link history.
6. Average the daily smirk within each firm-year window to get the final panel.

## Step 0. Setup

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wrds

In [ ]:
db = wrds.Connection(wrds_username=WRDS_USERNAME)

## Step 1. Explore the OptionMetrics library

Before writing a query against an unfamiliar database, always look at what is actually there. OptionMetrics' daily option price file is **partitioned by calendar year** — `opprcd1996`, `opprcd1997`, … — which is exactly why the SAS program needed a macro loop.

In [ ]:
assert "optionm" in db.list_libraries(), "OptionMetrics library not in your WRDS subscription."

In [ ]:
# The daily option price files, one table per calendar year
[t for t in db.list_tables(library="optionm") if "opprcd" in t]

In [ ]:
# The security master files, which we will need in Step 4 to get CUSIPs
[t for t in db.list_tables(library="optionm") if "sec" in t]

In [ ]:
db.describe_table(library="optionm", table="opprcd2022")

Note the row count: a *single* year of this table holds tens of millions of option-day observations. Downloading it and aggregating in pandas would be slow and would exhaust your laptop's memory. The whole point of Step 2 is that the `GROUP BY` happens **on the WRDS server**, so only one row per stock-day-type comes back.

## Step 2. Daily implied volatility of ATM calls and OTM puts

Read the filters one at a time — every one of them is a research design choice you should be able to defend:

| Filter | Why |
| --- | --- |
| `0.03 < impl_volatility < 2` | drops implausible IVs, which are usually the algorithm failing to converge on a stale or barely-traded quote |
| `open_interest > 0` | keeps only contracts investors actually hold; also serves as the aggregation weight |
| `volume is not missing` | drops contracts with no trading record for the day |
| `best_bid`, `best_offer` present and `best_offer >= best_bid` | drops crossed or one-sided quotes, i.e. corrupted price records |
| `delta` between **0.375 and 0.625** | **at-the-money calls** — a call struck at the money has a delta of about 0.5 |
| `delta` between **−0.375 and −0.125** | **out-of-the-money puts** — a put struck below the spot price has a small negative delta |


In [ ]:
def iv_query(year: int) -> str:
    """SQL for one year of open-interest-weighted IV, by security / day / option type."""
    return f"""
        SELECT secid,
               date,
               cp_flag,
               SUM(impl_volatility * open_interest) / SUM(open_interest) AS iv
        FROM optionm.opprcd{year}
        WHERE impl_volatility > 0.03
          AND impl_volatility < 2
          AND open_interest > 0
          AND volume IS NOT NULL
          AND best_bid IS NOT NULL
          AND best_offer IS NOT NULL
          AND best_offer >= best_bid
          AND ( (delta >  0.375 AND delta <  0.625)      -- at-the-money calls
             OR (delta > -0.375 AND delta < -0.125) )    -- out-of-the-money puts
        GROUP BY secid, date, cp_flag
    """

print(iv_query(2013))

### 2.1. Loop over the years, and cache each one

A long query you have to re-run because the kernel died is a bad research workflow. Instead we **cache** each year to a Parquet file (a compressed columnar format that round-trips dtypes exactly and is far smaller and faster than CSV for data this size). Re-running the loop then costs seconds rather than hours, and the cache is what makes the notebook reproducible.

Expect a few minutes per year on the first run.

In [ ]:
def get_year_iv(year: int, refresh: bool = False) -> pd.DataFrame:
    """Return one year of daily IV, querying WRDS only if it is not cached yet."""
    cache_file = f"../data/iv_{year}.parquet"

    if cache_file.exists() and not refresh:
        return pd.read_parquet(cache_file)

    t0 = time.perf_counter()
    df = db.raw_sql(iv_query(year), date_cols=["date"])

    # Shrink before caching: secid is a small integer and IVs need nothing like
    # float64 precision. Over 18 years this saves several GB of memory.
    df["secid"] = df["secid"].astype("int32")
    df["iv"] = df["iv"].astype("float32")

    df.to_parquet(cache_file, index=False)
    print(f"{year}: {len(df):>10,} rows  ({time.perf_counter() - t0:,.0f}s)")
    return df

In [ ]:
iv_all = pd.concat(
    [get_year_iv(year) for year in range(START_YEAR, END_YEAR + 1)],
    ignore_index=True,
)
print(f"{len(iv_all):,} security-day-type observations")
iv_all.head()

## Step 3. The daily volatility smirk

In [ ]:
daily = (
    iv_all
    .pivot(index=["secid", "date"], columns="cp_flag", values="iv")
    .rename(columns={"C": "atm_iv", "P": "otm_put_iv"})
    .rename_axis(columns=None)     # drop the leftover "cp_flag" name on the columns
    .reset_index()
    .dropna(subset=["atm_iv", "otm_put_iv"])   # keep stock-days with both legs
)

daily["iv_skew"] = daily["otm_put_iv"] - daily["atm_iv"]
daily.head()

`.pivot()` (rather than `.pivot_table()`) is deliberate: it raises an error if a `secid`-`date`-`cp_flag` combination appears twice, which would silently be averaged away by `pivot_table`. Getting an exception when your key is not unique is a feature.

A first sanity check — the smirk should be **positive on average**. That positive mean is the empirical fact the whole literature is built on: OTM puts are persistently more expensive, in IV terms, than ATM calls.

In [ ]:
daily[["atm_iv", "otm_put_iv", "iv_skew"]].describe()

## Step 4. From `secid` to `permno`

In [ ]:
# OptionMetrics security file: one row per secid, carrying its CUSIP
secid_cusip = db.raw_sql("SELECT secid, cusip AS ncusip FROM optionm.securd")
secid_cusip["secid"] = secid_cusip["secid"].astype("int32")

print(f"{len(secid_cusip):,} securities; secid unique: {secid_cusip['secid'].is_unique}")
secid_cusip.head()

In [ ]:
# CRSP name history: historical CUSIP with the window over which it was valid
stocknames = db.raw_sql(
    """
    SELECT permno, ncusip, namedt, nameenddt
    FROM crsp.stocknames
    WHERE ncusip IS NOT NULL
    """,
    date_cols=["namedt", "nameenddt"],
)
stocknames.head()

In [ ]:
merged = daily.merge(secid_cusip, on="secid", how="inner").merge(
    stocknames, on="ncusip", how="inner"
)

# Keep only matches where the option date falls inside the CUSIP's validity
# window. SAS writes an open-ended window as nameenddt=.E; here it is NULL/NaT.
in_window = (merged["date"] >= merged["namedt"]) & (
    (merged["date"] <= merged["nameenddt"]) | merged["nameenddt"].isna()
)

smirk_daily = merged.loc[in_window, ["permno", "date", "atm_iv", "iv_skew"]].copy()
smirk_daily["permno"] = smirk_daily["permno"].astype("int32")
smirk_daily = smirk_daily.sort_values(["permno", "date"]).reset_index(drop=True)

print(f"{len(smirk_daily):,} firm-day observations, {smirk_daily['permno'].nunique():,} permnos")
smirk_daily.head()

In [ ]:
# Diagnostic: a CUSIP that maps to two permnos over overlapping windows would
# produce duplicate firm-days and double-count in the averages below.
n_dup = smirk_daily.duplicated(subset=["permno", "date"]).sum()
print(f"duplicate permno-date rows: {n_dup:,}")

## Step 5. Compustat fundamentals and the CCM link

In [ ]:
comp = db.raw_sql(
    """
    SELECT gvkey, datadate, fyear, at, csho, prcc_f, dltt, ceq
    FROM comp.funda
    WHERE fyear >= 1995
      AND at IS NOT NULL
      AND indfmt = 'INDL'
      AND datafmt = 'STD'
      AND consol = 'C'
      AND popsrc = 'D'
    ORDER BY gvkey, datadate
    """,
    date_cols=["datadate"],
)
print(f"{len(comp):,} firm-years")
comp.head()

In [ ]:
# Build the controls in pandas, where a bad denominator is a NaN and not a crash.
# `.where(cond)` blanks out the values that fail the condition.
mktcap = comp["csho"] * comp["prcc_f"]          # market value of equity, $m

comp["firm_size"] = np.log(mktcap.where(mktcap > 0))
comp["leverage"] = comp["dltt"] / comp["at"].where(comp["at"] > 0)
comp["mb"] = mktcap / comp["ceq"].where(comp["ceq"] != 0)

comp[["firm_size", "leverage", "mb"]].describe()

In [ ]:
link = db.raw_sql(
    """
    SELECT gvkey, lpermno AS permno, linktype, linkprim, linkdt, linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LU', 'LC', 'LS')
    """,
    date_cols=["linkdt", "linkenddt"],
)
link.head()

In [ ]:
ccm = comp.merge(link, on="gvkey", how="inner")

valid_link = (
    (ccm["linkdt"] <= ccm["datadate"]) | ccm["linkdt"].isna()
) & ((ccm["datadate"] <= ccm["linkenddt"]) | ccm["linkenddt"].isna())

ccm = ccm.loc[valid_link].dropna(subset=["gvkey", "permno", "datadate"]).copy()
ccm["permno"] = ccm["permno"].astype("int32")
ccm["fyear"] = ccm["fyear"].astype("int16")

print(f"{len(ccm):,} linked firm-years, {ccm['permno'].nunique():,} permnos")
ccm.head()

## Step 6. Collapsing the daily smirk into a firm-year measure

In [ ]:
MONTHS_BEFORE, MONTHS_AFTER = 8, 3   # the SAS window: intck between -8 and 3


def month_index(s: pd.Series) -> pd.Series:
    """Months since year 0 — differencing this reproduces SAS's intck('month', ...)."""
    return s.dt.year * 12 + s.dt.month

In [ ]:
# 1. Daily -> permno-month sums and counts
smirk_daily["month"] = month_index(smirk_daily["date"])

monthly = (
    smirk_daily
    .groupby(["permno", "month"], as_index=False)
    .agg(sum_skew=("iv_skew", "sum"), sum_atm=("atm_iv", "sum"), n_days=("iv_skew", "size"))
)
print(f"{len(smirk_daily):,} firm-days -> {len(monthly):,} firm-months")
monthly.head()

In [ ]:
# 2. Each firm-year -> the 12 month keys that fall inside its window
firm_years = ccm[["gvkey", "permno", "datadate"]].drop_duplicates().copy()
firm_years["fy_month"] = month_index(firm_years["datadate"])

offsets = range(-MONTHS_BEFORE, MONTHS_AFTER + 1)
keys = pd.concat(
    [firm_years.assign(month=firm_years["fy_month"] + k) for k in offsets],
    ignore_index=True,
)
print(f"{len(firm_years):,} firm-years -> {len(keys):,} firm-year-month keys")
keys.head()

In [ ]:
# 3. Join on (permno, month), then re-aggregate to the firm-year
matched = keys.merge(monthly, on=["permno", "month"], how="inner")

firm_year_smirk = (
    matched
    .groupby(["gvkey", "permno", "datadate"], as_index=False)
    .agg(sum_skew=("sum_skew", "sum"), sum_atm=("sum_atm", "sum"), nob=("n_days", "sum"))
)

# Weighted by construction: total of the daily values / total number of days
firm_year_smirk["iv_skew"] = firm_year_smirk["sum_skew"] / firm_year_smirk["nob"]
firm_year_smirk["atm_iv"] = firm_year_smirk["sum_atm"] / firm_year_smirk["nob"]
firm_year_smirk = firm_year_smirk.drop(columns=["sum_skew", "sum_atm"])

firm_year_smirk.head()

In [ ]:
# Attach the smirk back to the Compustat/CCM panel. The inner join reproduces the
# SAS `delete from smirk1 where missing(IV_Skew) or missing(ATM_IV)`.
smirk = ccm.merge(firm_year_smirk, on=["gvkey", "permno", "datadate"], how="inner")

smirk = smirk[
    ["gvkey", "permno", "datadate", "fyear", "at",
     "firm_size", "leverage", "mb", "iv_skew", "atm_iv", "nob"]
].sort_values(["gvkey", "datadate"]).reset_index(drop=True)

print(f"{len(smirk):,} firm-years with a smirk, {smirk['gvkey'].nunique():,} firms")
smirk.head()

## Step 7. Check the result

In [ ]:
smirk[["iv_skew", "atm_iv", "nob", "firm_size", "leverage", "mb"]].describe()

In [ ]:
annual = smirk.groupby("fyear")["iv_skew"].mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(annual.index, annual.values, color="steelblue", linewidth=2, marker="o", markersize=5)

ax.set_title("Average option volatility smirk by fiscal year")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("IV skew (OTM put IV $-$ ATM call IV)")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

plt.show()